<a href="https://colab.research.google.com/github/suryasai99/Practice/blob/main/NN_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
x = [[1,2],[3,4],[5,6],[7,8]]
y = [[3],[7],[11],[15]]

In [4]:
# Defining the dataset class
class MyDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]

In [5]:
# defining the Dataloader object
ds = MyDataset(x, y)
dataloader = DataLoader(
    ds,
    batch_size = 2,
    shuffle = True
)

In [6]:
# define the model
model = nn.Sequential(nn.Linear(2,8),
                      nn.ReLU(),
                      nn.Linear(8,1)).to(device)

In [7]:
!pip install torch_summary

In [8]:
# summary of the model
from torchsummary import summary
summary(model, torch.zeros(1,2))

Layer (type:depth-idx)                   Output Shape              Param #
├─Linear: 1-1                            [-1, 8]                   24
├─ReLU: 1-2                              [-1, 8]                   --
├─Linear: 1-3                            [-1, 1]                   9
Total params: 33
Trainable params: 33
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00


Layer (type:depth-idx)                   Output Shape              Param #
├─Linear: 1-1                            [-1, 8]                   24
├─ReLU: 1-2                              [-1, 8]                   --
├─Linear: 1-3                            [-1, 1]                   9
Total params: 33
Trainable params: 33
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [9]:
# loss function
loss_func = nn.MSELoss()
from torch.optim import SGD
opt = SGD(model.parameters(), lr = 0.001)
import time
loss_history = []
start = time.time()
for _ in range(50):
    for ix, iy in dataloader:
        opt.zero_grad()
        loss_value = loss_func(model(ix), iy)
        loss_value.backward()
        opt.step()
        loss_history.append(loss_value.item())
end = time.time()
print(end - start)

0.25730419158935547


In [10]:
# models weights and biases
model.state_dict()

OrderedDict([('0.weight',
              tensor([[-0.0695, -0.0879],
                      [ 0.6857, -0.1824],
                      [ 0.1575, -0.5497],
                      [ 0.2518, -0.0608],
                      [ 0.0327,  0.9866],
                      [ 0.3192, -0.5059],
                      [ 0.9342,  0.3763],
                      [-0.0305,  0.2788]])),
             ('0.bias',
              tensor([-0.4324, -0.3083,  0.0010, -0.4284, -0.5402, -0.4221,  0.4650,  0.5833])),
             ('2.weight',
              tensor([[-0.2364,  0.1092, -0.0873,  0.1004,  0.6786,  0.1166,  0.9089,  0.0598]])),
             ('2.bias', tensor([0.1274]))])

In [11]:
# saving the model
torch.save(model.to('cpu').state_dict(), '/content/drive/MyDrive/Colab_Notebooks/Practise/sample_pytorch_model.pth')

In [12]:
# loading the model for validation
state_dict = torch.load('/content/drive/MyDrive/Colab_Notebooks/Practise/sample_pytorch_model.pth')
model.load_state_dict(state_dict)
model.to(device)

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

In [13]:
# validation set
val = [[8,9],[10,11],[1.5,2.5]]

In [14]:
model(torch.tensor(val).float().to(device))

tensor([[16.9181],
        [20.8617],
        [ 4.1218]], grad_fn=<AddmmBackward0>)